# Prompt Engineering with Groq API

**Prompt engineering** = writing the input (prompt) in a smart way so the LLM gives the output we want. We do not change the model, we only change the input.

LLM is a next-word predictor. A better prompt pushes it to predict better words.

- Bad prompt: `Write about dogs`
- Good prompt: `Write a 100-word paragraph on why dogs suit elderly people, warm tone`

**5 pillars of a good prompt**
1. Clarity - say exactly what you want
2. Context - give the background the model does not know
3. Specificity - fix format, length, tone
4. Iteration - first prompt rarely works, improve it again and again
5. Decomposition - break a big task into small steps

## test cell
Just checking that python / notebook is working.

In [1]:
print("hello world")

hello world


## import groq
Groq gives us a **free hosted LLM through an API**. We do not run the model on our laptop, we just send a request and get the answer back.
`Groq` is the class we use to talk to it.

In [1]:
from groq import Groq

## load api key from .env file
Never write the API key directly in code (anyone can steal it).
Keep it in a `.env` file as `GROQ_API_KEY=xxxx` and load it with `load_dotenv()`.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## os module
`os.environ` is used to read the environment variables (our key is stored there).

In [3]:
import os

## check the key
Print once to confirm the key is loaded. Do **not** share this output with anyone.

In [ ]:
print(os.environ["GROQ_API_KEY"])

## create client
`client` is our connection to Groq. Every request we make goes through this object.

In [5]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

## message format
We never send plain text to the model. We send a **list of messages**, and each message has:
- `role` - who is speaking
- `content` - what they said

| role | meaning |
|------|---------|
| system | rules for the whole chat (how to behave) |
| user | our question |
| assistant | answer given by the model |

In [10]:
message=[
        {
            "role": "user",
            "content": "EXPLAIN ATTENTION IS ALL YOU NEED PAPER",
        }
    ]

## Input prompt

This is the message (prompt) we send to the model. It is a list with one `user` message.
Output of the next cell just shows this list.

In [11]:
message

[{'role': 'user', 'content': 'EXPLAIN ATTENTION IS ALL YOU NEED PAPER'}]

## call the model
`client.chat.completions.create()` sends our messages to the model.
- `messages` - the input we send
- `model` - which LLM to use (here `openai/gpt-oss-20b`)

In [12]:
chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b")

## read the response
The model returns a big object. The actual answer text is at
`choices[0].message.content`.

In [13]:
print(chat_completion.choices[0].message.content)

## Attention Is All You Need – A “plain‑text” walk‑through  

**Citation**  
*Vaswani, A., Shazeer, N., Parmar, N., et al. 2017. “Attention Is All You Need.”*  
In *Advances in Neural Information Processing Systems* (NIPS 2017).  
This paper introduced the **Transformer** model, a neural network architecture that **replaces all recurrence and convolution with self‑attention** and became the foundation for the entire family of large language models (BERT, GPT, T5, etc.).

Below is a step‑by‑step explanation of what the paper does, why it matters, and how it works.

---

## 1. The Problem Space

| Traditional Seq2Seq Models | Limitations |
|----------------------------|-------------|
| Encoder‑decoder RNNs (LSTMs, GRUs) or CNNs | *Serial processing:* each time‑step depends on the previous one → slow training & inference. |
| *Attention* added on top of RNNs | Still sequential inside encoder/decoder; limited parallelism. |
| *Long‑range dependencies* are hard to learn; gradients vanish. |

## function to ask LLM
Writing the same code again and again is boring, so we put it in a function.
Now we only call `ask_llm("question")` and get the answer as text.

In [6]:
def ask_llm(user_query):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b")
    response = chat_completion.choices[0].message.content
    return response

In [7]:
ask_llm("tell m about me")

'I’m not actually able to “know” you in the way a person does—you’re the only one who truly knows yourself. What I can do is help you explore and discover more about yourself, whether that’s your strengths, interests, goals, or even just a quick self‑check‑in. Here are a few ways we could start:\n\n| Focus | Prompt | Why it helps |\n|-------|--------|--------------|\n| **Values** | What three values matter most to you when you make decisions? | Clarifying values gives a compass for life choices. |\n| **Strengths** | Think of a recent achievement. What skills did you use? | Identifying strengths builds confidence and skill mapping. |\n| **Goals** | Where would you like to be in 1, 5, or 10 years? | Setting goals frames a roadmap for growth. |\n| **Challenges** | What’s a recurring obstacle you face? | Recognizing patterns can spark solutions. |\n| **Curiosity** | What topics or hobbies always pull you in? | Passion can drive learning and career satisfaction. |\n\nIf you’d like, pick one

## chatbot with memory (multi-turn)
The model has **no memory**. It only knows what we send in the current request.
So to make a chatbot, we keep a `history` list and send the **full history every time**.

Steps in the loop:
1. take the user input
2. add it to history as `user`
3. send the whole history to the model
4. add the model reply to history as `assistant`
5. type `exit` to stop

The first message is the **system prompt**: it tells the model its job (here: coding assistant, and say "I do not know" for non-coding questions). It is set once and applies for the whole conversation.

In [28]:
history = []
system_prompt = {
        "role":"system", # user prompt
        "content":"You are AI coding assistant powered by github  . help me solve c , python , java , c++ coding problem. if user ask query unrelated to coding just tell them i do not know"
    }
history.append(system_prompt)
while True:
    user_query = input("Enter user query: ")
    if user_query.lower() == "exit":
        break
    message = {

        "role":"user", # user prompt
        "content":user_query
    }
    history.append(message)
    chat_completion = client.chat.completions.create( messages=history,model="openai/gpt-oss-20b")
    response = chat_completion.choices[0].message.content
    print(response)
    ai_response = {
        "role": "assistant",
        "content":response
    }
    history.append(ai_response)
    

Hello! 👋  
I’m here to help with C, Python, Java, or C++ coding questions. What can I assist you with today?
I’m ChatGPT, a large language‑model AI created by OpenAI. I’m here to help with coding questions, explanations, and a range of other topics. Let me know what you’d like to work on!


In [ ]:
system prompt , user prompt , ai response

In [27]:
history

[{'role': 'system',
  'content': 'You are AI coding assistant. help me solve c , python , java , c++ coding problem. if user ask query unrelated to coding just tell them i do not know'},
 {'role': 'user', 'content': 'hi'},
 {'role': 'assistant',
  'content': 'Hello! How can I help you with your coding questions today?'},
 {'role': 'user', 'content': 'tell me about indian constitution '},
 {'role': 'assistant',
  'content': 'I’m sorry, but I don’t have that information.'}]

## good prompt vs bad prompt
Same topic, two prompts. See how the answer changes.
- `prompt1` - short and vague, so the answer is long and random
- `prompt2` - gives **role + task + constraint** (expert, explain python, 5 points)

Rule: the clearer the prompt, the better the output.

In [18]:
prompt1  = """tell me about python?"""

In [19]:
prompt2 = """you are expert in python coding language.help me understand what is python and explain in 5 points."""

In [20]:
print(ask_llm(prompt1))

**Python** is a high‑level, interpreted programming language that emphasizes readability and productivity. It was created by Guido van Rossum and first released in 1991, and has since become one of the most popular languages in the world.

| Category | Highlights |
|----------|------------|
| **Syntax** | Minimal, English‑like syntax (e.g., `if`, `for`, `def`) – no semicolons, braces, or type declarations. |
| **Paradigms** | Multi‑paradigm: object‑oriented, functional, procedural, and even some meta‑programming. |
| **Typing** | Dynamically typed (but supports optional static typing via type hints & tools like `mypy`). |
| **Standard Library** | “Batteries included” – robust modules for networking, web, data, math, XML, databases, and more. |
| **Community & Ecosystem** | Huge ecosystem: PyPI hosts >400k packages (NumPy, pandas, Flask, Django, TensorFlow, etc.). |
| **Cross‑platform** | Works on Windows, macOS, Linux, embedded devices, and even mobile (via Kivy, BeeWare). |
| **Perfor

In [21]:
print(ask_llm(prompt2))

### What is Python?  
Python is a high‑level, general‑purpose programming language that emphasizes readability, simplicity, and rapid development. It was created by Guido van Rossum and first released in 1991, and it has since become one of the most widely used languages in areas ranging from web development to data science.

#### 5 Key Points About Python

1. **Readable, English‑like syntax**  
   - Python uses clear, indentation‑based blocks instead of braces or keywords like `end`.  
   - This makes code easier to read and maintain, especially for beginners.

2. **Interpreted, dynamically typed**  
   - Code runs line‑by‑line in a runtime interpreter.  
   - Variables are typed at runtime (`x = 5`, `x = "hello"`), which speeds up prototyping.

3. **Extensive Standard Library + Rich Ecosystem**  
   - The “batteries‑included” philosophy provides modules for file I/O, networking, web services, scientific computing, etc.  
   - Thousands of third‑party packages are available via PyPI (

## temperature and max_tokens
Two settings that control the output:

- **temperature** - randomness of the answer
  - `0` = same, safe, predictable answer every time
  - `1` = more creative, different answer each time
- **max_tokens** - maximum length of the answer. If the limit is too small, the answer gets **cut in the middle**.

(1 token is roughly 3/4 of a word)

In [45]:
def ask_llm_temp_1(user_query,max_token = 300):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b",temperature=1,max_tokens=max_token)
    response = chat_completion.choices[0].message.content
    return response

In [ ]:
def ask_llm_temp_0(user_query):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b",temperature=0,max_tokens=)
    response = chat_completion.choices[0].message.content
    return response

## compare temperature 1 vs 0
Run the same query on both functions. Temp 1 gives a different story each run, temp 0 gives almost the same story each run.

In [26]:
query  = "Write a story about python in 5 points only"

In [41]:
print(ask_llm_temp_1(query))

1. In the emerald depths of the rainforest, a young python named Slyther slid silently across the riverbank, feeling the pulse of the earth beneath his scales.  
2. Curiosity tugged at him when he discovered an abandoned compass, its needle spinning wildly as if eager for adventure.  
3. Guided by the compass, he ventured beyond familiar mangrove thickets into a sun‑lit clearing where a silver kite fluttered, its tail tangled in vines.  
4. The python wrapped his body around the kite’s string, gently untying the knots and freeing it; in gratitude, the kite gifted Slyther a feathered message of friendship.  
5. From that day on, Slyther became a guardian of the forest’s secrets, proving that even the quietest creatures can hold a compass that leads to wonder.


In [50]:
print(ask_llm_temp_0(query))

**A Python Tale in Five Points**

1. **Discovery** – In a cramped office, Maya stumbles upon a dusty book titled *“Python: The Language of Simplicity.”* She opens it, and the first line, “Hello, World!” sparks a curiosity that feels like a secret handshake with the future.  

2. **First Script** – She writes a tiny program to calculate the Fibonacci sequence, watching the numbers cascade on her screen. Each line of code feels like a brushstroke, turning abstract logic into a living, breathing pattern.  

3. **The Bug** – When she runs the script, an error pops up: *“IndexError: list index out of range.”* Panic flickers across her face, but she remembers the book’s advice: “Read the error, then read the code.”  

4. **Resolution** – Maya traces the stack trace, discovers an off‑by‑one mistake, and corrects it. The program runs flawlessly, and the Fibonacci numbers roll out like a well‑tuned orchestra.  

5. **Celebration** – She closes the laptop, takes a deep breath, and smiles. Python

## final ask_llm function
Now one function handles everything. `max_token` and `temperature` are optional, we can change them whenever we want.
Default: `temperature = 0`, `max_token = 1000`.

In [69]:
def ask_llm(user_query,max_token = 1000,temperature = 0):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b",temperature=temperature,max_tokens=max_token)
    response = chat_completion.choices[0].message.content
    return response

In [53]:
print(query)

Write a story about python in 5 points only


## effect of max_tokens
- `max_token=200` - answer is short and may stop in the middle
- `max_token=2000` - answer is complete

More tokens = more cost and more time, so set it only as much as needed.

In [63]:
print(ask_llm(user_query=query,max_token=200))

**A Python Tale in Five Points**

1. **Discovery** – In a cramped office, Maya stumbles upon a dusty book titled *“Python: The Language of Sim


In [64]:
print(ask_llm(user_query=query,max_token=2000))

**A Python Tale in Five Points**

1. **Discovery** – In a cramped office, Maya stumbles upon a dusty book titled *“Python: The Language of Simplicity.”* She opens it, and the first line, “Hello, World!” sparks a curiosity that feels like a secret handshake with the future.  

2. **First Script** – She writes a tiny program to calculate the Fibonacci sequence, watching the numbers cascade on her screen. Each line of code feels like a brushstroke, turning abstract logic into a living, breathing pattern.  

3. **The Bug** – When she runs the script, an error pops up: *“IndexError: list index out of range.”* Panic flickers across her face, but she remembers the book’s advice: “Read the error, then read the code.”  

4. **Resolution** – Maya traces the stack trace, discovers an off‑by‑one mistake, and corrects it. The program runs flawlessly, and the Fibonacci numbers roll out like a well‑tuned orchestra.  

5. **Celebration** – She closes the laptop, takes a deep breath, and smiles. Python

## template place holder

**Prompt template** = a reusable prompt where only some words change.
First we try hard-coded prompts (transformer, bert), then convert them into one template.

In [66]:
PROMPT = """you are expert in summarizing topic. the user is 10th class student. only give 5 point summary. explain about transformer"""

In [70]:
print(ask_llm(PROMPT))

**Transformer – 5‑point summary**

1. **Self‑attention mechanism**  
   - Each word (or token) looks at every other word in the sentence to decide how much it should “pay attention” to them.  
   - This lets the model capture long‑range relationships without needing to read the sentence word‑by‑word.

2. **Layered encoder–decoder structure**  
   - The encoder turns the input text into a set of rich, context‑aware vectors.  
   - The decoder uses those vectors to generate output (e.g., translated words), also applying attention to the encoder’s output.

3. **Parallel processing**  
   - Unlike older recurrent models that read words one after another, transformers process all words at once.  
   - This speeds up training and allows the use of powerful GPUs.

4. **Position encoding**  
   - Because the model doesn’t read sequentially, it adds a “position code” to each word so it knows the order of words in the sentence.

5. **Foundation for modern NLP models**  
   - Variants like BERT, 

In [71]:
PROMPT = """you are expert in summarizing topic. the user is 10th class student. only give 5 point summary. explain about bert"""

In [72]:
print(ask_llm(PROMPT))

**BERT (Bidirectional Encoder Representations from Transformers)** – 5‑point summary for a 10th‑class student  

1. **What it is**  
   - BERT is a *pre‑trained language model* created by Google that learns how words relate to each other in a sentence.  
   - It uses the *Transformer* architecture, which focuses on attention mechanisms to capture context from both left and right sides of a word (hence “bidirectional”).

2. **How it learns**  
   - During pre‑training, BERT is given two tasks:  
     1. **Masked Language Modeling (MLM)** – some words are hidden, and the model must guess them.  
     2. **Next Sentence Prediction (NSP)** – the model decides if one sentence logically follows another.  
   - These tasks teach BERT a deep understanding of grammar, facts, and relationships between words.

3. **Fine‑tuning for specific jobs**  
   - After pre‑training, BERT can be *fine‑tuned* on a small, task‑specific dataset (e.g., sentiment analysis, question answering).  
   - Only a few 

## template with placeholders
Writing a new prompt every time is slow. Instead we make **one template** and keep blanks `{ }` for the parts that change.
It is just like a function: fixed instruction + changing inputs.

Here the placeholders are `{user_type}`, `{number}`, `{topic}`. We fill them with `.format()`.
Real apps (chatbots, RAG) fill templates with code like this.

In [73]:
PROMPT = """you are expert in summarizing topic. the user is {user_type}. only give {number} point summary. explain about {topic}"""

In [ ]:
PROMPT.format(number = 3, topic = "GPT", user_type= "professional")

'you are expert in summarizing topic. the user is professional. only give 3 point summary. explain about GPT'

In [75]:
print(ask_llm(PROMPT.format(number = 3, topic = "GPT", user_type= "professional")))

**GPT (Generative Pre‑trained Transformer) – 3‑point overview**

1. **Architecture & Pre‑training**  
   - GPT is a transformer‑based language model that uses a stack of self‑attention layers to capture long‑range dependencies.  
   - It is first trained on massive, diverse text corpora in an unsupervised, autoregressive fashion (predicting the next token), which endows it with a broad linguistic and factual base.

2. **Fine‑tuning & Adaptation**  
   - After pre‑training, GPT can be fine‑tuned on domain‑specific data or via instruction‑following paradigms (e.g., RLHF) to align outputs with user intent, safety constraints, and task‑specific performance.  
   - This two‑stage pipeline allows rapid deployment across varied applications—from code generation to medical triage—while preserving the core language understanding.

3. **Practical Impact & Limitations**  
   - GPT powers conversational agents, content creation, summarization, and more, delivering near‑human fluency and contextual

In [76]:
PROMPT.format(number = 5, topic = "photosynthesis", user_type= "10th class student")

'you are expert in summarizing topic. the user is 10th class student. only give 5 point summary. explain about photosynthesis'

In [77]:
print(ask_llm(PROMPT.format(number = 5, topic = "photosynthesis", user_type= "10th class student")))

**Photosynthesis – 5‑Point Summary for 10th‑Grade Students**

1. **What it is**  
   - A chemical process that converts light energy into chemical energy.  
   - Plants, algae, and some bacteria use it to make glucose (sugar) from carbon dioxide (CO₂) and water (H₂O).

2. **Where it happens**  
   - Inside chloroplasts, the green organelles in plant cells.  
   - The outer membrane of the chloroplast contains chlorophyll, the pigment that captures sunlight.

3. **The two main stages**  
   - **Light‑dependent reactions** (occurs in the thylakoid membranes): sunlight excites electrons, producing ATP and NADPH while splitting water to release O₂.  
   - **Calvin cycle (light‑independent reactions)** (occurs in the stroma): ATP and NADPH drive the fixation of CO₂ into glucose.

4. **Key outputs**  
   - **Glucose** – the energy storage molecule used by the plant for growth and metabolism.  
   - **Oxygen** – released as a by‑product, essential for most life on Earth.

5. **Why it matters*

# Extra theory notes (from the PPT)

## prompt anatomy
A good prompt usually has these parts:

| part | question it answers |
|------|---------------------|
| Role | who should the model act as? |
| Context | what background does it need? |
| Task | what exactly to do? |
| Input data | on what content? |
| Constraints | format, length, tone |
| Output format | how should the answer look? |
| Examples (optional) | sample input -> output |

Example:
- Role: experienced biology teacher
- Context: class 10 board exam student
- Task: explain photosynthesis
- Constraints: simple language, max 150 words
- Output format: 3 bullets + 1 line summary

## prompt anatomy example

In [ ]:
role = "You are an experienced high school biology teacher."
context = "The student is preparing for a Class 10 board exam."
task = "Explain photosynthesis."
constraints = "Use simple language, max 150 words, no jargon."
output_format = "Give 3 bullet points and 1 line summary."

In [ ]:
PROMPT = f"{role}\n{context}\n{task}\n{constraints}\n{output_format}"
print(PROMPT)

In [ ]:
print(ask_llm(PROMPT))

## zero-shot, one-shot, few-shot
How many examples we put inside the prompt:

- **Zero-shot** - no example, only instruction (simple tasks)
- **One-shot** - one example (to show the format)
- **Few-shot** - 2 to 5 examples (when pattern / tone must match exactly)

```
Classify sentiment as Positive / Negative / Neutral.
Review: "Amazing service!" -> Positive
Review: "It was okay." -> Neutral
Review: "The movie was a waste of time." ->
```
The model learns the pattern from examples, **no training needed**. More examples = more tokens = more cost, and after 4-5 examples there is little benefit.

## zero shot
Only instruction, **no example**.

In [8]:
zero_shot = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: The movie was a waste of time.
Sentiment:"""

In [9]:
print(ask_llm(zero_shot))

Negative


## one shot
Give **one example** so the model learns the format.

In [11]:
one_shot = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: Amazing service!
Sentiment: Positive

Review: {user_query}
Sentiment:"""

In [12]:
input = one_shot.format(user_query = "The product quality is terrible.")

In [13]:
print(ask_llm(one_shot))

Please provide the review text you would like classified.


## few shot
Give **2 to 5 examples** (cover every label) so the model copies the pattern.

In [28]:
few_shot = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: Amazing service!
Sentiment: Positive

Review: It was okay, nothing special.
Sentiment: Neutral

Review: Worst experience ever.
Sentiment: Negative

Review: {user_query}
Sentiment:"""

In [32]:
user_query = "The delivery was late but the food was great."

In [35]:
few_shot.format(user_query = user_query)

'Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.\n\nReview: Amazing service!\nSentiment: Positive\n\nReview: It was okay, nothing special.\nSentiment: Neutral\n\nReview: Worst experience ever.\nSentiment: Negative\n\nReview: The delivery was late but the food was great.\nSentiment:'

In [36]:
input=few_shot.format(user_query = user_query)

In [37]:
print(ask_llm(input))

Positive


## few shot template
Same few shot prompt but with a `{review}` placeholder, so we can test many reviews.

In [ ]:
FEW_SHOT_PROMPT = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: Amazing service!
Sentiment: Positive

Review: It was okay, nothing special.
Sentiment: Neutral

Review: Worst experience ever.
Sentiment: Negative

Review: {review}
Sentiment:"""

In [ ]:
reviews = ["I loved this phone!", "Product is average.", "Totally disappointed, money wasted."]

for r in reviews:
    print(r, "->", ask_llm(FEW_SHOT_PROMPT.format(review=r)))

## few shot using messages
Another way: put each example as a `user` message and the correct answer as an `assistant` message.

In [ ]:
few_shot_messages = [
    {"role": "system", "content": "Classify the sentiment as Positive, Negative or Neutral. Reply with the label only."},
    {"role": "user", "content": "Amazing service!"},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It was okay, nothing special."},
    {"role": "assistant", "content": "Neutral"},
    {"role": "user", "content": "Worst experience ever."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "The movie was a waste of time."},
]

In [ ]:
chat_completion = client.chat.completions.create(messages=few_shot_messages, model="openai/gpt-oss-20b", temperature=0)
print(chat_completion.choices[0].message.content)

## system prompt vs user prompt
- **System prompt** = how to behave, always (does not change)
- **User prompt** = what to do right now (changes every request)

| goes in System | goes in User |
|----------------|--------------|
| Role / persona | Task (specific question) |
| General instructions | Input data |
| Constraints (tone, length) | Retrieved context (RAG chunks) |
| Output format | |

Simple test: **will it change? No -> System. Yes -> User.**

Persona = *who* the model is (voice and style). System prompt = the *container* of all the rules. Persona is written inside the system prompt.

## ask with system prompt
New function that takes a **system prompt** (fixed rules) and a **user query** (changes every time).

In [ ]:
def ask_llm_system(system_prompt, user_query, max_token=1000, temperature=0):
    message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]
    chat_completion = client.chat.completions.create(messages=message, model="openai/gpt-oss-20b", temperature=temperature, max_tokens=max_token)
    return chat_completion.choices[0].message.content

In [ ]:
SYSTEM_PROMPT = """You are an experienced high school biology teacher.
Always explain in simple language, no jargon.
Answer in 3 bullet points + 1 line summary. Max 150 words."""

USER_PROMPT = "Explain {topic} to a student preparing for {exam_context}."

In [ ]:
print(ask_llm_system(SYSTEM_PROMPT, USER_PROMPT.format(topic="photosynthesis", exam_context="Class 10 board exam")))

In [ ]:
print(ask_llm_system(SYSTEM_PROMPT, USER_PROMPT.format(topic="respiration", exam_context="Class 10 board exam")))

## rag style user prompt
Context that **changes every query** (retrieved chunks) goes in the user prompt.

In [ ]:
RAG_SYSTEM = "You are a helpful assistant. Answer only using the given context. If the answer is not in the context, say Not found."

RAG_USER = """Context: {retrieved_chunks}

Question: {user_question}"""

In [ ]:
chunks = "HubNine is an AI consulting company. It builds RAG based solutions for clients."

print(ask_llm_system(RAG_SYSTEM, RAG_USER.format(retrieved_chunks=chunks, user_question="What does HubNine do?")))
print(ask_llm_system(RAG_SYSTEM, RAG_USER.format(retrieved_chunks=chunks, user_question="Who is the CEO of HubNine?")))

## prompt refinement
First prompt is rarely perfect. Keep improving:

write prompt -> run -> check output -> find the problem -> change prompt -> repeat

| problem | fix |
|---------|-----|
| answer too long | add a length limit |
| wrong format | give a template or example |
| made-up facts (hallucination) | "use only the given context, say I don't know if unsure" |
| tone changes | add persona + tone rule |
| no reasoning shown | "think step by step" |
| instruction ignored | move it to the end of the prompt |

## refinement example
Start with a simple prompt, check the output, then improve it step by step.

In [ ]:
v1 = "Explain machine learning"
print(ask_llm(v1))

Too long -> **add a length constraint**.

In [ ]:
v2 = "Explain machine learning in maximum 3 sentences"
print(ask_llm(v2))

Add **role + audience + format**.

In [ ]:
v3 = """You are a friendly teacher.
Explain machine learning to a 10th class student.
Give exactly 3 bullet points, each under 15 words.
Then give 1 real life example."""
print(ask_llm(v3))

## fix hallucination
Model may make up answers. Tell it to use only the given context.

In [ ]:
context = "HubNine is an AI consulting company that builds RAG based solutions."

bad_prompt = "Who is the CEO of HubNine?"

good_prompt = f"""Answer only using the context below. If the answer is not in the context, say "I don't know".

Context: {context}

Question: Who is the CEO of HubNine?"""

In [ ]:
print(ask_llm(bad_prompt))
print("-----")
print(ask_llm(good_prompt))

## chain of thought
Add **think step by step** when the problem needs reasoning.

In [ ]:
math_prompt = "A shop sells pens at 12 rupees each. Riya buys 7 pens and pays with a 100 rupee note. How much change does she get? Think step by step."
print(ask_llm(math_prompt))

## structured output (JSON)
Apps need output that code can read, not free text. So we ask for JSON with fixed keys.

```
Extract as JSON with keys "name", "age", "city".
Text: "John is 29 and lives in Mumbai."
-> {"name": "John", "age": 29, "city": "Mumbai"}
```
Tips: give the exact schema and say "no extra text, no markdown".
In real projects we use JSON mode, function/tool calling, or Pydantic to force valid output.

## json output using prompt

In [ ]:
import json

In [ ]:
json_prompt = """Extract the details as JSON with keys "name", "age", "city".
Output must be only JSON, no markdown and no extra text.

Text: John is 29 and lives in Mumbai."""

In [ ]:
output = ask_llm(json_prompt)
print(output)
print(type(output))

Output is still a **string**. Convert it to a python dictionary with `json.loads()`.

In [ ]:
data = json.loads(output)
print(data)
print(data["name"], data["age"], data["city"])

## json mode
API forces valid JSON. The word **JSON** must be present in the prompt.

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[{"role": "user", "content": json_prompt}],
    model="openai/gpt-oss-20b",
    temperature=0,
    response_format={"type": "json_object"}
)
result = json.loads(chat_completion.choices[0].message.content)
print(result)

## explicit schema
Give the exact structure we want.

In [ ]:
schema_prompt = """Extract details from the text. Output must strictly follow this JSON schema:
{
  "name": "string",
  "age": "integer",
  "city": "string",
  "skills": ["string"]
}
No markdown formatting or extra text.

Text: Priya is 24, she lives in Pune and knows python and sql."""

print(json.loads(ask_llm(schema_prompt)))

## task based prompts (6 common templates)
1. **Summarization** - summarize in {N} sentences, focus on {aspect}
2. **Question answering** - answer only from the context, else say "Not found"
3. **Information extraction** - extract {entities} as a JSON list
4. **Classification** - choose one of {labels}, reply with label only
5. **Translation** - translate {source} to {target}, keep the tone
6. **Code generation** - write a {language} function for {task} with docstring and example

## six task prompts
Same sample text, different task templates.

In [ ]:
text = "Mumbai is the financial capital of India. Rahul moved there from Delhi in 2020 to work at a bank. He loves the street food but finds the monsoon traffic very stressful."

In [ ]:
tasks = {
    "summarization": "Summarize the following text in {n} sentences, focusing on {aspect}.\n\nText: {text}",
    "question answering": "Answer only using the context. If the answer is not present, say Not found.\n\nContext: {text}\n\nQuestion: {question}",
    "information extraction": "Extract all {entity_type} from the text as a JSON list. Output only JSON.\n\nText: {text}",
    "classification": "Classify the text into one of {category_list}. Respond with the label only.\n\nText: {text}",
    "translation": "Translate the following text from {source_lang} to {target_lang}. Preserve the tone.\n\nText: {text}",
    "code generation": "Write a {language} function to {task_description}. Add a docstring and one example.",
}

In [ ]:
print(ask_llm(tasks["summarization"].format(n=1, aspect="Rahul's life", text=text)))

In [ ]:
print(ask_llm(tasks["question answering"].format(text=text, question="Where did Rahul work?")))
print(ask_llm(tasks["question answering"].format(text=text, question="What is Rahul's salary?")))

In [ ]:
print(ask_llm(tasks["information extraction"].format(entity_type="city names", text=text)))

In [ ]:
print(ask_llm(tasks["classification"].format(category_list="[Travel, Finance, Sports, Food]", text=text)))

In [ ]:
print(ask_llm(tasks["translation"].format(source_lang="English", target_lang="Hindi", text=text)))

In [ ]:
print(ask_llm(tasks["code generation"].format(language="python", task_description="check whether a number is prime")))

## important API parameters 
| parameter | use |
|-----------|-----|
| model | which model to call |
| messages | chat history (system / user / assistant) |
| max_tokens | limit of output length |
| temperature | 0 = fixed, 1 = creative |
| top_p | picks words from top probable group only |
| stop | words where the model must stop |
| stream | show the answer word by word |

Cost and speed depend on **input tokens + output tokens**. Longer prompt = more cost and slower reply.

**Flow:** prompt -> tokenizer -> model -> tokens generated -> detokenizer -> text answer

## top_p and stop
- `top_p` - model picks next word only from the top probable words (lower = safer)
- `stop` - model stops when this word comes

In [ ]:
def ask_llm_full(user_query, max_token=1000, temperature=0, top_p=1, stop=None):
    message = [{"role": "user", "content": user_query}]
    chat_completion = client.chat.completions.create(
        messages=message,
        model="openai/gpt-oss-20b",
        temperature=temperature,
        max_tokens=max_token,
        top_p=top_p,
        stop=stop
    )
    return chat_completion.choices[0].message.content

In [ ]:
print(ask_llm_full("Write numbers from 1 to 10, one number per line"))
print("-----")
print(ask_llm_full("Write numbers from 1 to 10, one number per line", stop=["5"]))

## token usage (cost)
Bill = input tokens + output tokens.

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[{"role": "user", "content": "Explain few-shot prompting in 2 lines."}],
    model="openai/gpt-oss-20b"
)
print(chat_completion.choices[0].message.content)
print(chat_completion.usage)

## streaming
`stream=True` gives the answer **word by word** (like ChatGPT typing) instead of waiting for the full answer.

In [ ]:
stream = client.chat.completions.create(
    messages=[{"role": "user", "content": "Explain few-shot prompting in 2 lines."}],
    model="openai/gpt-oss-20b",
    stream=True
)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")